# 🛡️ AI-Driven Phishing Email Detection Using NLP
## IICT Summer Internship – Project 2

---
**Objective:** Build and compare four machine-learning classifiers (Naive Bayes, Logistic Regression, Random Forest, MLP Neural Network) that detect phishing emails using TF-IDF text features combined with 12 hand-crafted structural/metadata features.

**Dataset:** Phishing Email Dataset  
- Columns: `Email Text`, `Email Type` (Safe Email / Phishing Email)  
- ~18,650 labelled emails

| Section | Content |
|---------|--------|
| §1 | Data Loading & EDA |
| §2 | Preprocessing & Feature Engineering |
| §3 | Model Training (4 classifiers) |
| §4 | Evaluation & Comparative Analysis |


In [ ]:
import os, sys, warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import scipy.sparse as sp

from sklearn.model_selection      import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes          import MultinomialNB, ComplementNB
from sklearn.linear_model         import LogisticRegression
from sklearn.ensemble             import RandomForestClassifier
from sklearn.neural_network       import MLPClassifier
from sklearn.preprocessing        import MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
import download_data
from metadata_features import EmailMetadataExtractor, extract_metadata_matrix

warnings.filterwarnings('ignore')

# ── Dark theme ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor' : '#0d0d1a',
    'axes.facecolor'   : '#111127',
    'axes.edgecolor'   : '#1a1a3e',
    'text.color'       : '#e8e8ff',
    'axes.labelcolor'  : '#e8e8ff',
    'xtick.color'      : '#e8e8ff',
    'ytick.color'      : '#e8e8ff',
    'axes.titlecolor'  : '#00d4ff',
    'axes.grid'        : True,
    'grid.color'       : '#1a1a3e',
    'grid.alpha'       : 0.4,
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 11,
})
CYAN   = '#00d4ff'
ORANGE = '#ff6b35'
GREEN  = '#39ff14'
PURPLE = '#b44be1'
os.makedirs('plots', exist_ok=True)
print('✅ All imports successful.')

---
# §1 – Data Loading & Exploratory Data Analysis


In [ ]:
DATA_PATH = download_data.download()
df = pd.read_csv(DATA_PATH)
df.rename(columns={'Email Text': 'text', 'Email Type': 'label'}, inplace=True)
df.drop(columns=[c for c in df.columns if 'Unnamed' in c], inplace=True)
df.dropna(subset=['text'], inplace=True)
df['text'] = df['text'].astype(str)

# Encode label: 1 = Phishing, 0 = Safe
df['label_int'] = (df['label'] == 'Phishing Email').astype(int)

print(f'Shape : {df.shape}')
print(f'Labels:\n{df["label"].value_counts()}')
df.head()

In [ ]:
# ── EDA 1: Class distribution ─────────────────────────────────────────────────
counts = df['label'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Class Distribution – Phishing vs. Safe Email', fontsize=15, color=CYAN)

bars = axes[0].bar(counts.index, counts.values, color=[GREEN, ORANGE], edgecolor='white', lw=0.6)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
                 f'{val:,}', ha='center', va='bottom', fontsize=11)
axes[0].set_title('Email Count by Class')

axes[1].pie(counts.values, labels=counts.index, colors=[GREEN, ORANGE],
            autopct='%1.1f%%', startangle=90, textprops={'color': 'white'})
axes[1].set_title('Class Balance')

plt.tight_layout()
plt.savefig('plots/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── EDA 2: Email text length ──────────────────────────────────────────────────
df['word_count'] = df['text'].str.split().str.len()

fig, ax = plt.subplots(figsize=(12, 5))
ax.set_title('Email Length Distribution by Class', color=CYAN, fontsize=13)

for label_val, color, name in [('Safe Email', GREEN, 'Safe'), ('Phishing Email', ORANGE, 'Phishing')]:
    subset = df[df['label'] == label_val]['word_count'].clip(0, 800)
    ax.hist(subset, bins=60, alpha=0.7, color=color, label=name)

ax.set_xlabel('Word Count')
ax.set_ylabel('Number of Emails')
ax.legend()
plt.tight_layout()
plt.savefig('plots/email_length_dist.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── EDA 3: Extract metadata features for EDA ──────────────────────────────────
print('Extracting metadata features …')
ext = EmailMetadataExtractor()
meta_sample = df.head(3000).copy()
for feat in ext.FEATURE_NAMES:
    meta_sample[feat] = meta_sample['text'].apply(lambda t: ext.extract(t)[feat])

# Exclamation count per class
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Metadata Feature Distributions (sample 3000)', fontsize=13, color=CYAN)

for ax, feat, title in [
    (axes[0], 'exclamation_count', 'Exclamation Marks'),
    (axes[1], 'urgent_word_count', 'Urgency Words'),
    (axes[2], 'uppercase_ratio',   'Uppercase Word Ratio'),
]:
    for label_val, color, name in [('Safe Email', GREEN, 'Safe'), ('Phishing Email', ORANGE, 'Phishing')]:
        subset = meta_sample[meta_sample['label'] == label_val][feat]
        ax.hist(subset.clip(0, 20), bins=20, alpha=0.7, color=color, label=name)
    ax.set_title(title)
    ax.set_xlabel(feat)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('plots/metadata_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

---
# §2 – Preprocessing & Feature Engineering


In [ ]:
import re

def clean_email(text: str) -> str:
    """Basic email-specific text cleaning."""
    text = re.sub(r'https?://\S+', ' URL ', text)    # replace URLs with token
    text = re.sub(r'\S+@\S+', ' EMAIL ', text)       # replace emails
    text = re.sub(r'<[^>]+>', ' ', text)             # strip HTML
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)        # remove non-alpha
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    return text

print('Cleaning email text …')
df['cleaned'] = df['text'].apply(clean_email)
print(f'Sample cleaned email:\n  {df["cleaned"].iloc[0][:150]} …')

In [ ]:
# ── Train / Test split ────────────────────────────────────────────────────────
X_text = df['cleaned'].values
X_raw  = df['text'].values
y      = df['label_int'].values

(
  X_text_train, X_text_test,
  X_raw_train,  X_raw_test,
  y_train,      y_test
) = train_test_split(
    X_text, X_raw, y,
    test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_text_train):,}  |  Test: {len(X_text_test):,}')
print(f'Train balance: {Counter(y_train)}')

In [ ]:
# ── TF-IDF features ───────────────────────────────────────────────────────────
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2), sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(X_text_train)     # sparse
X_test_tfidf  = tfidf.transform(X_text_test)

print(f'TF-IDF train : {X_train_tfidf.shape}')

# ── Metadata features ─────────────────────────────────────────────────────────
print('Extracting metadata from training emails …')
M_train = extract_metadata_matrix(list(X_raw_train))
M_test  = extract_metadata_matrix(list(X_raw_test))

# Scale metadata
scaler  = MinMaxScaler()
M_train_scaled = scaler.fit_transform(M_train)
M_test_scaled  = scaler.transform(M_test)

# ── Concatenate TF-IDF + metadata ─────────────────────────────────────────────
X_train_combined = sp.hstack([X_train_tfidf, sp.csr_matrix(M_train_scaled)])
X_test_combined  = sp.hstack([X_test_tfidf,  sp.csr_matrix(M_test_scaled)])
print(f'Combined train matrix : {X_train_combined.shape}')

---
# §3 – Model Training


In [ ]:
def evaluate(name, model, X_tr, X_te, y_tr, y_te, use_dense=False):
    print(f'Training [{name}] …', end=' ', flush=True)
    Xtr = X_tr.toarray() if use_dense and sp.issparse(X_tr) else X_tr
    Xte = X_te.toarray() if use_dense and sp.issparse(X_te) else X_te
    model.fit(Xtr, y_tr)
    y_pred = model.predict(Xte)
    m = {
        'Model'    : name,
        'Accuracy' : accuracy_score(y_te, y_pred),
        'Precision': precision_score(y_te, y_pred, zero_division=0),
        'Recall'   : recall_score(y_te, y_pred, zero_division=0),
        'F1-Score' : f1_score(y_te, y_pred, zero_division=0),
    }
    print(f"F1 = {m['F1-Score']:.4f}")
    return model, y_pred, m

results = []
models  = {}
preds   = {}

In [ ]:
# ── Model 1: Naive Bayes ──────────────────────────────────────────────────────
nb = ComplementNB(alpha=0.1)
m, pred, met = evaluate('Naive Bayes', nb, X_train_tfidf, X_test_tfidf, y_train, y_test)
models['NB'] = m; preds['NB'] = pred; results.append(met)
print(classification_report(y_test, pred, target_names=['Safe','Phishing']))

In [ ]:
# ── Model 2: Logistic Regression ──────────────────────────────────────────────
lr = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs', n_jobs=-1)
m, pred, met = evaluate('Logistic Regression', lr, X_train_combined, X_test_combined, y_train, y_test)
models['LR'] = m; preds['LR'] = pred; results.append(met)
print(classification_report(y_test, pred, target_names=['Safe','Phishing']))

In [ ]:
# ── Model 3: Random Forest ────────────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
m, pred, met = evaluate('Random Forest', rf, X_train_combined, X_test_combined, y_train, y_test)
models['RF'] = m; preds['RF'] = pred; results.append(met)
print(classification_report(y_test, pred, target_names=['Safe','Phishing']))

In [ ]:
# ── Model 4: MLP Neural Network ───────────────────────────────────────────────
mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64), activation='relu',
    solver='adam', max_iter=30, random_state=42,
    early_stopping=True, validation_fraction=0.1,
)
m, pred, met = evaluate('MLP Neural Network', mlp,
                         X_train_combined, X_test_combined,
                         y_train, y_test, use_dense=True)
models['MLP'] = m; preds['MLP'] = pred; results.append(met)
print(classification_report(y_test, pred, target_names=['Safe','Phishing']))

---
# §4 – Evaluation & Comparative Analysis


In [ ]:
results_df = pd.DataFrame(results).set_index('Model').round(4)
print('\n===  Model Comparison  ===')
results_df

In [ ]:
# ── Metric comparison bar chart ───────────────────────────────────────────────
metrics_cols = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics_cols))
width = 0.18
colors_ = [CYAN, ORANGE, GREEN, PURPLE]

fig, ax = plt.subplots(figsize=(14, 6))
ax.set_title('Phishing Email Detection – Model Comparison', fontsize=14, color=CYAN)

for i, (name, color) in enumerate(zip(results_df.index, colors_)):
    vals = results_df.loc[name, metrics_cols].values
    bars = ax.bar(x + i*width, vals, width, label=name, color=color, alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + width*1.5)
ax.set_xticklabels(metrics_cols)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score')
ax.legend()
plt.tight_layout()
plt.savefig('plots/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Confusion matrices ────────────────────────────────────────────────────────
model_labels = [('NB','Naive Bayes'),('LR','Logistic Regression'),
                ('RF','Random Forest'),('MLP','MLP Neural Network')]

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Confusion Matrices – All Models', fontsize=15, color=CYAN)

for ax, (key, label) in zip(axes.flatten(), model_labels):
    cm = confusion_matrix(y_test, preds[key])
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Blues', linewidths=0.5,
                xticklabels=['Safe','Phishing'], yticklabels=['Safe','Phishing'])
    ax.set_title(label)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('plots/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── ROC Curves ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
ax.set_title('ROC Curves – Phishing Detection', fontsize=14, color=CYAN)
colors_ = [CYAN, ORANGE, GREEN, PURPLE]

for (key, label), color in zip(model_labels, colors_):
    mod = models[key]
    Xte = X_test_combined
    if key == 'NB':
        Xte = X_test_tfidf
    if hasattr(mod, 'predict_proba'):
        if sp.issparse(Xte) and key == 'MLP':
            proba = mod.predict_proba(Xte.toarray())[:, 1]
        else:
            proba = mod.predict_proba(Xte)[:, 1]
    else:
        proba = preds[key].astype(float)
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, lw=2, color=color, label=f'{label}  (AUC={auc(fpr,tpr):.3f})')

ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('plots/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Metadata feature importance (Random Forest) ───────────────────────────────
from metadata_features import EmailMetadataExtractor
meta_feat_names = EmailMetadataExtractor.FEATURE_NAMES

all_feat_names = list(tfidf.get_feature_names_out()) + meta_feat_names

importances = models['RF'].feature_importances_

# Top 20 overall
top_idx = importances.argsort()[-20:][::-1]
top_names = [all_feat_names[i] if i < len(all_feat_names) else f'feat_{i}' for i in top_idx]

fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(list(reversed(top_names)), list(reversed(importances[top_idx])), color=CYAN)
ax.set_title('Random Forest – Top 20 Feature Importances', color=CYAN, fontsize=13)
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('plots/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Save best model ───────────────────────────────────────────────────────────
import pickle
os.makedirs('models', exist_ok=True)

best_name = results_df['F1-Score'].idxmax()
key_map   = {'Naive Bayes':'NB','Logistic Regression':'LR',
              'Random Forest':'RF','MLP Neural Network':'MLP'}
best_key  = key_map[best_name]

with open('models/best_model.pkl', 'wb') as f:
    pickle.dump({
        'vectorizer': tfidf,
        'scaler'    : scaler,
        'model'     : models[best_key],
        'name'      : best_name,
    }, f)

print(f'Best model: {best_name}  →  saved to models/best_model.pkl')
print('\n🏁 Project 2 notebook complete!')